In [1]:
import duckdb
import polars as pl

print("🚀 Iniciando el Motor Analítico DuckDB...")
con = duckdb.connect(database=':memory:')

# Ajustes de hilos y RAM para tu máquina
con.execute("PRAGMA threads=4;")          
con.execute("PRAGMA memory_limit='4GB';")  
print("✅ Conexión establecida en memoria.")

🚀 Iniciando el Motor Analítico DuckDB...
✅ Conexión establecida en memoria.


In [2]:
print("📦 Creando vistas virtuales sobre los archivos Parquet...")

con.execute("CREATE VIEW fact_presupuesto AS SELECT * FROM '../data/03_gold/fact_presupuesto.parquet';")
con.execute("CREATE VIEW dim_geografia AS SELECT * FROM '../data/03_gold/dim_geografia.parquet';")
con.execute("CREATE VIEW dim_institucion AS SELECT * FROM '../data/03_gold/dim_institucion.parquet';")
con.execute("CREATE VIEW dim_programatica AS SELECT * FROM '../data/03_gold/dim_programatica.parquet';")

print("✅ Vistas creadas. Los archivos están listos para ser consultados.")

📦 Creando vistas virtuales sobre los archivos Parquet...
✅ Vistas creadas. Los archivos están listos para ser consultados.


# Reportes Gerenciales Analiticos
Este cuaderno ejecuta las consultas analiticas de alto rendimiento sobre la capa Gold utilizando DuckDB en memoria.

## Reporte 1: Top 5 Departamentos con Mayor Gasto Real (Devengado) en 2024
A continuacion, se inicializa el motor analitico y se presenta el analisis del gasto devengado agrupado por departamento, utilizando los nombres oficiales y omitiendo registros nulos o vacios.

In [3]:
import duckdb
import polars as pl

print("Iniciando el Motor Analitico DuckDB...")
con = duckdb.connect(database=':memory:')

# Configuracion de rendimiento de la sesion
con.execute("PRAGMA threads=4;")          
con.execute("PRAGMA memory_limit='4GB';")  
print("Conexion establecida en memoria.")

print("Mapeando archivos base para el Reporte 1...")
con.execute("CREATE VIEW fact_presupuesto AS SELECT * FROM '../data/03_gold/fact_presupuesto.parquet';")
con.execute("CREATE VIEW dim_geografia AS SELECT * FROM '../data/03_gold/dim_geografia.parquet';")
print("Vistas listas.")

# Consulta del Reporte 1 utilizando nombres oficiales
query_top_regiones = """
SELECT 
    d_geo.departamento_ejecutora_nombre AS departamento,
    SUM(f.monto) / 1000000000 AS millones_soles
FROM fact_presupuesto f
JOIN dim_geografia d_geo ON f.sk_geografia_id = d_geo.sk_geografia_id
WHERE f.ano_eje = 2024 
  AND f.fase = 'devengado'
  AND d_geo.departamento_ejecutora_nombre IS NOT NULL 
  AND TRIM(d_geo.departamento_ejecutora_nombre) != ''
GROUP BY 1
ORDER BY 2 DESC
LIMIT 5;
"""

print("Ejecutando Reporte 1...")
df_reporte_1 = con.execute(query_top_regiones).pl()
print(df_reporte_1)

Iniciando el Motor Analitico DuckDB...
Conexion establecida en memoria.
Mapeando archivos base para el Reporte 1...
Vistas listas.
Ejecutando Reporte 1...
shape: (5, 2)
┌──────────────┬────────────────┐
│ departamento ┆ millones_soles │
│ ---          ┆ ---            │
│ str          ┆ f64            │
╞══════════════╪════════════════╡
│ lima         ┆ 91.426836      │
│ cusco        ┆ 5.627538       │
│ piura        ┆ 4.549765       │
│ arequipa     ┆ 3.984606       │
│ cajamarca    ┆ 3.892488       │
└──────────────┴────────────────┘


In [4]:
print("Mapeando dimensiones adicionales para Reportes 2 y 3...")
con.execute("CREATE VIEW dim_institucion AS SELECT * FROM '../data/03_gold/dim_institucion.parquet';")
con.execute("CREATE VIEW dim_programatica AS SELECT * FROM '../data/03_gold/dim_programatica.parquet';")
print("Dimensiones listas.")

# Consulta del Reporte 2: Evolucion del PIM
query_evolucion = """
SELECT 
    ano_eje AS ano, 
    SUM(monto) / 1000000000 AS pim_total_miles_de_millones
FROM fact_presupuesto 
WHERE fase = 'pim' 
GROUP BY 1 
ORDER BY 1 ASC;
"""

print("Ejecutando Reporte 2...")
df_reporte_2 = con.execute(query_evolucion).pl()
print(df_reporte_2)

# Consulta del Reporte 3: Analisis por Sectores de Gobierno
query_obras = """
SELECT 
    d_inst.sector_nombre AS sector_gobierno,
    APPROX_COUNT_DISTINCT(d_prog.producto_proyecto) AS aprox_cantidad_obras,
    SUM(f.monto) / 1000000000 AS gasto_en_millones
FROM fact_presupuesto f
JOIN dim_institucion d_inst ON f.sk_institucion_id = d_inst.sk_institucion_id
JOIN dim_programatica d_prog ON f.sk_programatica_id = d_prog.sk_programatica_id
WHERE f.ano_eje = 2024 
  AND f.fase = 'devengado'
  AND d_inst.sector_nombre IS NOT NULL 
  AND TRIM(d_inst.sector_nombre) != ''
GROUP BY 1 
ORDER BY 3 DESC 
LIMIT 5;
"""

print("Ejecutando Reporte 3...")
df_reporte_3 = con.execute(query_obras).pl()
print(df_reporte_3)

Mapeando dimensiones adicionales para Reportes 2 y 3...
Dimensiones listas.
Ejecutando Reporte 2...
shape: (5, 2)
┌──────┬─────────────────────────────┐
│ ano  ┆ pim_total_miles_de_millones │
│ ---  ┆ ---                         │
│ i32  ┆ f64                         │
╞══════╪═════════════════════════════╡
│ 2022 ┆ 147.056421                  │
│ 2023 ┆ 157.339402                  │
│ 2024 ┆ 167.036237                  │
│ 2025 ┆ 172.067633                  │
│ 2026 ┆ 172.067633                  │
└──────┴─────────────────────────────┘
Ejecutando Reporte 3...
shape: (5, 3)
┌──────────────────────┬──────────────────────┬───────────────────┐
│ sector_gobierno      ┆ aprox_cantidad_obras ┆ gasto_en_millones │
│ ---                  ┆ ---                  ┆ ---               │
│ str                  ┆ i64                  ┆ f64               │
╞══════════════════════╪══════════════════════╪═══════════════════╡
│ gobiernos regionales ┆ 5730                 ┆ 541.80959         │
│ economia 